In [ ]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

client = Groq(api_key="GROQ_API_KEY")

MODEL = "llama-3.1-8b-instant"

In [ ]:
# -----------------------------
# Utility: LLM Call
# -----------------------------
def call_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a systems engineering assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

In [ ]:
# -----------------------------
# Agent 1: Actor Extractor
# -----------------------------
def extract_actors(requirements):
    prompt = f"""
Extract all actors interacting with the system.

Rules:
- Include humans and external systems
- Exclude internal components
- Return JSON only

Requirements:
{requirements}

Output format:
{{
  "actors": []
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

In [ ]:
# -----------------------------
# Agent 2: Use Case Generator
# -----------------------------
def generate_use_cases(requirements, actors):
    prompt = f"""
Extract system use cases.

Steps:
1. Use the provided actors
2. Identify their goals
3. Convert each goal into a use case
4. Avoid duplicates

Actors:
{json.dumps(actors, indent=2)}

Requirements:
{requirements}

Output format:
{{
  "use_cases": [
    {{
      "name": "",
      "actor": "",
      "description": "",
      "preconditions": [],
      "postconditions": []
    }}
  ]
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

In [ ]:
# -----------------------------
# Agent 3: Validator
# -----------------------------
def validate_use_cases(requirements, actors, use_cases):
    prompt = f"""
Validate and improve the use cases.

Tasks:
1. Add missing use cases
2. Remove duplicates
3. Ensure all interactions are covered
4. Ensure correct actor mapping

Actors:
{json.dumps(actors, indent=2)}

Use Cases:
{json.dumps(use_cases, indent=2)}

Requirements:
{requirements}

Return ONLY final JSON:
{{
  "use_cases": []
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

In [ ]:
# -----------------------------
# Pipeline Runner
# -----------------------------
def run_pipeline(requirements):
    print("🔹 Extracting actors...")
    actors = extract_actors(requirements)

    print("🔹 Generating use cases...")
    use_cases = generate_use_cases(requirements, actors)

    print("🔹 Validating use cases...")
    final_use_cases = validate_use_cases(
        requirements, actors, use_cases
    )

    return {
        "actors": actors["actors"],
        "use_cases": final_use_cases["use_cases"]
    }

In [ ]:
# -----------------------------
# Example Usage
# -----------------------------
if __name__ == "__main__":
    requirements = """
    The system allows users to register and log in.
    Users can browse products and place orders.
    Admin can manage inventory.
    Payment is handled by an external payment gateway.
    """

    result = run_pipeline(requirements)

    print("\n✅ Final Output:")
    print(json.dumps(result, indent=2))